# Welcome to Week 5 - Agent Frameworks

## Day 2: AWS Strands

Same week, same five steps, a new framework. The whole idea this week is that once you know one agent framework, you basically know them all, so every day we build the same agent the same five steps and watch the idioms rhyme:

1. **Create the agent** - give it a model and a system prompt.
2. **Run it** - send a message, get a reply.
3. **Add tools** - plain typed functions the agent can call.
4. **Add MCP** - connect a tool server someone else wrote, wired the same way every time.
5. **Put it in a loop with a goal** - hand it an objective and let it work, step after step, until the job is done.

Steps 1 and 2 are still just an LLM call. Tools and MCP give the agent things it can do. Step 5 is where it becomes an agent: the framework runs the loop on its own, choosing a tool, reading the result, choosing again, until the goal is met.

The running project is the same SQLite todo board from Day 1. A worker picks one goal off the board, plans its own steps, does the work with its agent loop, and ticks each step off. The board code (`board.py`) is the same file, byte for byte; only the framework around it changes.

Today is **AWS Strands**: a lightweight, open-source SDK built around a model-driven loop. You hand the agent a task and some tools, and the model itself plans, calls a tool, reads the result and keeps going until it is done. Strands is known for two things: that minimal loop, and true portability, where you write the tools once and swap the backing model in a single line. We will see that swap at the end.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The AWS Strands docs</h2>
            <span style="color:#00bfff;">The docs live at <a href="https://strandsagents.com">https://strandsagents.com</a> and are clean and short. Strands ships a new release most weeks, so we pin the version (1.43.0 here) and treat older blog posts with care, since several APIs moved recently. One thing to watch: a bare <code>Agent()</code> with no model defaults to AWS Bedrock, so we always pass a model.</span>
        </td>
    </tr>
</table>

## Setup

Two things this day needs, both already in place from earlier weeks:

- **Node**, for `npx` (the filesystem MCP server runs over it). Check with `node --version`.
- **`OPENAI_API_KEY`** in the repo-root `.env`. Today's frameworks use OpenAI's `gpt-5.4-mini`, and that key has been in your `.env` since Week 1.

Strands is part of the repo's environment, so a normal `uv sync` from the repo root installs everything. Open this notebook in Cursor and pick the repo's default **Python 3.12.12** kernel, the same one you use every week, then run the cells top to bottom.

Warm the filesystem MCP server once now so the first run is fast, then stop it with Ctrl-C as soon as it prints that it is running:

```bash
npx -y @modelcontextprotocol/server-filesystem .
```

In [1]:
import os
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from strands import Agent, tool
from strands.models.openai import OpenAIModel
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters

load_dotenv(override=True)

True

## Step 1: Create the agent

In Strands an agent is an `Agent`: a model plus a `system_prompt`. We build the model once as an `OpenAIModel` pointed at `gpt-5.4-mini` and reuse it everywhere. One thing to know up front: a bare `Agent()` with no model quietly defaults to AWS Bedrock and expects AWS credentials, so we always pass `model=` explicitly.

In [4]:
# 사용할 모델 식별자를 상수로 지정
MODEL = "gpt-5.4-mini"

# OpenAI 모델 객체 생성
# client_args: OpenAI 클라이언트에 전달할 인자 (환경 변수에서 API 키를 읽어 전달)
# model_id: 위에서 지정한 모델 식별자
model = OpenAIModel(client_args={"api_key": os.environ["OPENAI_API_KEY"]}, model_id=MODEL)

# 에이전트 생성
# model: 위에서 만든 모델 객체를 사용
# system_prompt: 에이전트의 응답 방식(말투, 길이)을 지정하는 시스템 프롬프트
agent = Agent(
    model=model,
    system_prompt="당신은 간결하고 친근한 어시스턴트입니다. 한 문장의 짧은 답변으로 응답하세요.",
)

## Step 2: Run it

Send a message and await the reply. Strands streams the text as it is generated, so you watch it appear live; with no tools yet there is nothing to loop over, so the agent just answers. This is still only an LLM call. A plain `agent("...")` works too, but inside a notebook we await `invoke_async` so it cooperates with Jupyter's event loop.

In [5]:
# 에이전트를 비동기로 호출하여 입력 문구에 대한 응답을 받음
# invoke_async: 비동기 실행 메서드이므로 await로 결과를 기다림
# 반환값 result에는 에이전트의 응답 결과가 담김
result = await agent.invoke_async("스페인어로 인사해줘.")

¡Hola!

## Our project this week: a SQLite todo board

The worker coordinates through the same tiny SQLite board as Day 1, the same `board.py` file: one file, one table, no server to run. A worker is handed one **goal**; to reach it, it writes its own **step** todos under that goal, ticks each one off as it goes, and closes the goal at the end. Under the hood the board is just a list of dicts (a goal has `parent_id` of None, a step points at its goal):

In [6]:
import board

# 기존 보드를 지우고 비어 있는 새 보드를 생성
board.reset_board()

# 최상위 목표 하나를 보드에 추가
# 내용: notes.txt를 읽고, 자연스러운 스페인어로 번역한 뒤, 그 결과를 spanish.txt에 쓴다
board.add_goal("notes.txt를 읽고, 그 내용을 자연스러운 스페인어로 번역한 뒤, 스페인어 결과를 spanish.txt에 써라.")

# 보드에 있는 모든 할 일(목표와 단계)을 dict 리스트로 조회
# 셀의 마지막 표현식이므로 Jupyter에서는 반환값이 그대로 출력됨
board.list_todos()

[{'id': 1,
  'parent_id': None,
  'task': 'notes.txt를 읽고, 그 내용을 자연스러운 스페인어로 번역한 뒤, 스페인어 결과를 spanish.txt에 써라.',
  'status': 'pending',
  'result': ''}]

`show_board()` prints that same data nicely, in the rich style from Week 1: each goal with its steps indented beneath, done tasks struck through in green and the one in progress in yellow. There are no steps yet; the worker writes its own when it plans.

In [7]:
board.show_board()

Goal #1: notes.txt를 읽고, 그 내용을 자연스러운 스페인어로 번역한 뒤, 스페인어 결과를 spanish.txt에 써라.

## Step 3: Add tools

A tool in Strands is a typed Python function wrapped in the `@tool` decorator. The type hints become the argument schema, the docstring becomes the description the model reads (its `Args:` section documents each parameter), and you hand the functions to the agent in its `tools=[...]` list.

We write three small board tools: `show_todos` to read the board, `plan_steps` to break a goal into steps, and `complete_task` to mark a todo done. Here we give a quick agent just two of them and ask what is on the board; watch it decide, on its own, to call `show_todos` before it answers. That decide, call, read, answer is the agent loop starting to turn. All three tools come together in step 5.

In [8]:
@tool
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    # 보드의 모든 할 일을 dict 리스트로 반환
    return board.list_todos()

@tool
def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board.

    Args:
        goal_id: The id of the goal to break down.
        steps: Short descriptions of the steps to take, in order.
    """
    # 리스트 컴프리헨션으로 steps를 순서대로 순회하며 board.add_step을 호출
    # 각 호출이 새로 만들어진 단계의 id를 반환하므로 step_ids는 생성된 단계 id 목록이 됨
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

@tool
def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) done and record a short result summary.

    Args:
        task_id: The id of the todo to mark done.
        result: A short summary of what was accomplished.
    """
    # 해당 할 일의 상태를 'done'으로 바꾸고 결과 요약을 기록
    board.complete_todo(task_id, result)
    # 처리한 id와 상태를 dict로 반환해 모델이 결과를 확인할 수 있게 함
    return {"task_id": task_id, "status": "done"}

In [9]:
# 보드 관리용 에이전트 생성
# model: 앞서 만든 모델 객체를 사용
# system_prompt: 에이전트의 역할을 지정하는 시스템 프롬프트
# tools: 에이전트가 호출할 수 있는 도구 목록
board_agent = Agent(
    model=model,
    system_prompt="당신은 공유 할 일 보드를 관리하는 것을 돕습니다.",
    tools=[show_todos, complete_task],
)

In [10]:
# 보드 에이전트를 비동기로 호출
# 입력 문구: 지금 보드에 무엇이 있고 그 상태는 어떤지 묻는 질문
# 이 질문에 답하려면 에이전트가 show_todos 도구를 호출해야 함
# 반환값 result에는 도구 호출을 거쳐 만들어진 최종 응답이 담김
result = await board_agent.invoke_async("지금 보드에 무엇이 있고, 각각의 상태는 어떤가요?")


Tool #1: show_todos
현재 보드에는 1개의 할 일이 있습니다.

1. **notes.txt를 읽고, 그 내용을 자연스러운 스페인어로 번역한 뒤, 스페인어 결과를 spanish.txt에 써라.**
   - 상태: **pending**

원하시면 이 작업을 진행하거나, 완료된 항목이 있으면 상태를 갱신해드릴게요.

## Step 4: Add MCP

MCP is just more tools: ones you did not write, connected over a small protocol. We give the agent the filesystem reference server, the same Node server every framework uses this week, scoped to a single `workspace` folder so it can only touch files in there. In Strands you wrap the server in an `MCPClient` and add it to the same `tools=[...]` list, and Strands manages the connection lifecycle for you.

We pass `errlog=subprocess.DEVNULL` to the server. That quiets its startup logging, and it is also what lets the server run from a Jupyter kernel on Windows, where the kernel's stderr has no real file descriptor. On Mac and Linux it changes nothing.

In [11]:
# the only folder the agent may touch
# 에이전트가 접근할 수 있는 유일한 폴더
# Path("workspace")는 상대 경로이므로 resolve()로 절대 경로로 변환
workspace = Path("workspace").resolve()

# MCP 파일시스템 서버에 연결하는 클라이언트 생성
# 첫 번째 인자는 실제 연결을 만들어내는 팩토리 함수 (lambda로 지연 실행)
filesystem = MCPClient(
    # stdio_client: 표준 입출력(stdin/stdout)으로 통신하는 MCP 서버에 연결
    lambda: stdio_client(
        StdioServerParameters(
            # command: 서버를 실행할 명령어. npx로 Node 패키지를 실행
            command="npx",
            # args: npx에 전달할 인자
            #   -y : 패키지 설치 여부를 묻지 않고 자동 진행
            #   @modelcontextprotocol/server-filesystem : 실행할 MCP 파일시스템 서버 패키지
            #   str(workspace) : 서버가 접근을 허용할 디렉터리 경로
            args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
            # start the server in the workspace so relative file names resolve there
            # cwd: 서버 프로세스의 작업 디렉터리를 workspace로 지정
            # 이렇게 하면 "notes.txt" 같은 상대 파일명이 workspace 기준으로 해석됨
            cwd=str(workspace),
        ),
        # errlog: 서버가 stderr로 출력하는 로그를 버림 (노트북 출력이 지저분해지는 것을 방지)
        errlog=subprocess.DEVNULL,
    ),
    # startup_timeout: 서버가 준비될 때까지 기다리는 최대 시간(초)
    # npx가 패키지를 처음 내려받는 경우 시간이 걸리므로 넉넉하게 60초로 설정
    startup_timeout=60
)

In [12]:
# 파일 작업용 에이전트 생성
# model: 앞서 만든 모델 객체를 사용
# system_prompt: 워크스페이스 안의 파일을 읽고 쓸 수 있다는 역할과 도구 사용 지시
# tools: MCP 파일시스템 클라이언트를 그대로 전달 (서버가 제공하는 도구들이 에이전트에 등록됨)
file_agent = Agent(
    model=model,
    system_prompt="당신은 워크스페이스 안의 파일을 읽고 쓸 수 있습니다. 요청받은 일을 하기 위해 도구를 사용하세요.",
    tools=[filesystem],
)

In [13]:
# 파일 에이전트를 비동기로 호출
# 입력 문구: notes.txt를 읽고 한 문장으로 짧게 요약하라는 요청
# 이 요청을 처리하려면 에이전트가 MCP 파일시스템 서버의 파일 읽기 도구를 호출해야 함
# 파일명이 상대 경로이므로 앞서 cwd로 지정한 workspace 폴더를 기준으로 해석됨
result = await file_agent.invoke_async("notes.txt를 읽고 한 문장으로 짧게 요약해줘.")


Tool #1: read_text_file
작은 언어 튜터를 팀원들과 함께 하나씩 완성해 가는 작업 지침입니다.

## Step 5: Put it in a loop with a goal

Now the payoff. Give one agent all three board tools and the filesystem server, hand it the goal, and let it run. It plans its own steps on the board, works them with its file tools, ticks each one off, and closes the goal when the work is done. That is the agent loop running on its own: read, plan, act, check off, repeat. Watch the board fill with steps and then strike them through.

In [15]:
# 워커 에이전트에게 줄 시스템 프롬프트
# 역할, 작업 순서(계획 수립 → 실행 → 단계 완료 표시 → 목표 종료), 작업 폴더 제한을 지시
INSTRUCTIONS = """
당신은 공유 할 일 보드와 파일 도구 모음을 가진 신중한 작업자입니다.

대기 중인 목표를 맡아 끝까지 완수하세요. 먼저 짧은 계획을 세우는 것으로 시작하세요. 그 작업이 실제로 쪼개지는 몇 개의 구체적인 단계들을 보드의 목표 아래에 추가하면 됩니다. 그런 다음 파일 도구로 그것들을 수행하고, 각 단계를 끝낼 때마다 완료로 표시하세요. 단계가 모두 끝나면 목표를 닫으세요. 당신의 파일은 도구가 접근할 수 있는 단 하나의 폴더 안에 있습니다.
"""

# 워커 에이전트 생성
# tools: 보드 조작 도구 3개와 MCP 파일시스템 클라이언트를 함께 전달
#   show_todos    - 보드 조회 (목표 id와 상태 확인용)
#   plan_steps    - 목표를 단계들로 분해해 보드에 등록
#   complete_task - 단계 또는 목표를 완료 처리
#   filesystem    - 파일 읽기/쓰기 도구 제공
worker = Agent(
    model=model,
    system_prompt=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task, filesystem],
)

# 기존 보드를 지우고 새 보드를 생성
board.reset_board()

# 최상위 목표를 추가하고 그 id를 받아둠
goal_id = board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.")

# 목표 상태를 'in_progress'로 바꿔, 워커가 이 일을 집어 들었음을 표시
board.claim_todo(goal_id)

# 워커를 비동기로 호출해 보드의 대기 중인 목표를 처리하도록 지시
# 이 호출 안에서 에이전트가 도구들을 반복 호출하며 계획 수립과 실행을 진행
await worker.invoke_async("Please work the pending goal on the board.")

# 작업이 끝난 뒤 보드 상태를 사람이 보기 좋은 형태로 출력
# 완료된 항목은 초록색 취소선, 진행 중인 항목은 노란색으로 표시됨
board.show_board()

먼저 보드의 대기 중인 목표를 확인하고, 필요한 경우 짧은 실행 계획을 세운 뒤 파일 작업을 진행하겠습니다.
Tool #1: show_todos
목표를 3단계로 나누겠습니다: 1) notes.txt 읽기, 2) 내용을 자연스러운 스페인어로 번역하기, 3) spanish.txt로 저장하기.
Tool #2: plan_steps

Tool #3: read_text_file
번역을 완료한 뒤 파일에 저장하겠습니다.
Tool #4: write_file

Tool #5: complete_task

Tool #6: complete_task

Tool #7: complete_task

Tool #8: complete_task
Done — I translated `notes.txt` into Spanish and saved it as `spanish.txt`, then closed the goal on the board.

Goal #1: Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.  Completed reading, translating, and saving the Spanish version to spanish.txt.
  Step #2: Read notes.txt.  Read notes.txt successfully.
  Step #3: Translate the contents into natural Spanish.  Translated the notes into natural Spanish.
  Step #4: Write the Spanish text to spanish.txt.  Wrote the Spanish translation to spanish.txt.

## Run the same worker from the terminal

Everything you just watched in step 5 is also packaged as a small script, `strands_worker.py`, sitting next to this notebook. It seeds the same goal, builds the same agent with the same three board tools and the filesystem MCP server, and runs the same loop, only from the command line instead of the kernel. Open a terminal in this folder and run it:

```bash
uv run strands_worker.py
```

You will see the agent plan its steps, work the goal and tick each one off, then the finished board and the Spanish it wrote. This is the shape every worker takes on Day 5, where a Google ADK orchestrator launches one of these per framework as a parallel subprocess against a single shared board.

## The one cool thing: swap the model in one line

Strands' headline feature is portability. You wrote the tools and the loop once. To run the exact same agent on a different provider, you change one line, the `OpenAIModel`. Point `base_url` at any OpenAI-compatible endpoint and nothing else moves:

```python
# The same agent on OpenRouter, Ollama, LM Studio, a local vLLM server, anything OpenAI-compatible
model = OpenAIModel(
    client_args={
        "api_key": OPENROUTER_KEY,             # any non-empty string for a local server
        "base_url": "https://openrouter.ai/api/v1",
    },
    model_id="gpt-5.4-mini",                    # or whatever that endpoint serves
)
```

Strands also speaks to Anthropic, Gemini and Bedrock through their own model classes, but the OpenAI-compatible `base_url` covers most of the ecosystem. The tools, the filesystem MCP server and the board never change. That is the whole thesis of the week inside a single framework: the agent is the agent, and the model behind it is a swappable detail.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Seed a different goal on the board, for example "write a short haiku about Madrid into madrid.txt", and run the worker again. Does it plan sensible steps and pick the right file tools? Then try the one-line model swap: point the <code>OpenAIModel</code> at another OpenAI-compatible endpoint you have access to, rerun the worker, and watch the same agent run on a different model.</span>
        </td>
    </tr>
</table>